In [ ]:
import numpy as np
import sep
from astropy.io import fits
import matplotlib.pyplot as plt
from matplotlib import rcParams
from astropy.table import Table
from matplotlib.patches import Ellipse

%matplotlib inline

rcParams['figure.figsize'] = [10., 8.]

In [ ]:
input_file = "./input.fits" # oda:POSIXPath

### sep.extract()
thresh = 1.5 # oda:Float
err_option = 'float_globalrms' # oda:String; oda:allowed_value 'float_globalrms','array_rms'
# gain = None
# mask = None
minarea = 5 # oda:Integer
# filter_kernel=default_kernel
filter_type = 'matched' # oda:String; oda:allowed_value 'matched','conv'
deblend_nthresh = 32 # oda:Integer
deblend_cont = 0.005 # oda:Float
clean = True # oda:Boolean ; oda:allowed_value True,False
clean_param = 1.0 # oda:Float
# segmentation_map=False

### sep.Background()
# mask = None
# maskthresh = 0.0
bw = 64 # oda:Integer
bh = 64 # oda:Integer
fw = 3 # oda:Integer
fh = 3 # oda:Integer
fthresh = 0.0 # oda:Float

In [ ]:
hdul = fits.open(input_file)
data = hdul[0].data
data = data.astype(data.dtype.newbyteorder('='))

In [ ]:
# measure a spatially varying background on the image
bkg = sep.Background(data, mask=None, maskthresh=0.0, bw=bw, bh=bh, fw=fw, fh=fh, fthresh=fthresh)

# evaluate background as 2-d array, same size as original image
bkg_image = bkg.back()

# evaluate the background noise as 2-d array, same size as original image
bkg_rms = bkg.rms()

# subtract the background
data_sub = data - bkg

if err_option == 'float_globalrms':
    err = bkg.globalrms
else:
    err = bkg.rms()

# extract sources:
objects = sep.extract(data_sub, thresh, 
                      err=err,
                      gain=None,
                      mask=None, 
                      minarea=minarea, 
                      filter_type=filter_type, 
                      deblend_nthresh=deblend_nthresh, 
                      deblend_cont=deblend_cont, 
                      clean=clean, 
                      clean_param=clean_param)

In [ ]:
# show the background
fig, ax = plt.subplots()
im = ax.imshow(bkg_image, interpolation='nearest', cmap='gray', origin='lower')
fig.colorbar(im)
fig.savefig('bkg_image.png',format='png',bbox_inches='tight')

# show the background noise
fig, ax = plt.subplots()
im = ax.imshow(bkg_rms, interpolation='nearest', cmap='gray', origin='lower')
fig.colorbar(im)
fig.savefig('bkg_rms.png',format='png',bbox_inches='tight')

# plot image
fig, ax = plt.subplots()
m, s = np.mean(data), np.std(data)
im = ax.imshow(data, interpolation='nearest', cmap='gray',
               vmin=m-s, vmax=m+s, origin='lower')
fig.colorbar(im)
fig.savefig('fits2image.png',format='png',bbox_inches='tight')

# plot background-subtracted image
fig, ax = plt.subplots()
m, s = np.mean(data_sub), np.std(data_sub)
im = ax.imshow(data_sub, interpolation='nearest', cmap='gray',
               vmin=m-s, vmax=m+s, origin='lower')

# plot an ellipse for each object
for i in range(len(objects)):
    e = Ellipse(xy=(objects['x'][i], objects['y'][i]),
                width=6*objects['a'][i],
                height=6*objects['b'][i],
                angle=objects['theta'][i] * 180. / np.pi)
    e.set_facecolor('none')
    e.set_edgecolor('red')
    ax.add_artist(e)

fig.savefig('sources.png',format='png',bbox_inches='tight')

plt.show()

In [ ]:
from oda_api.data_products import PictureProduct, ODAAstropyTable

bin_bkg_image = PictureProduct.from_file('bkg_image.png')
bin_rms_image = PictureProduct.from_file('bkg_rms.png')
bin_dat_image = PictureProduct.from_file('fits2image.png')
bin_sor_image = PictureProduct.from_file('sources.png')

cat =  ODAAstropyTable(Table(data=objects))

In [ ]:
picture1 = bin_bkg_image # oda:ODAPictureProduct
picture2 = bin_rms_image # oda:ODAPictureProduct
picture3 = bin_dat_image # oda:ODAPictureProduct
picture4 = bin_sor_image # oda:ODAPictureProduct
catalog_table = cat # oda:ODAAstropyTable